# Layer 1 — Descriptive Pass

Automated overview of all indicators in the project's processed data.

**What this notebook produces:**
- Univariate profiles (distribution, min/max/median, missing %, outliers)
- Time series trends (direction, rate of change, trend lines)
- Changepoint detection (sharp shifts, year and magnitude)
- Pairwise correlations (Pearson and Spearman, flagged above threshold)
- Geographic variation (if geo-level data exists)

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pipeline.config import load_config, get_data_processed_dir

# --- Set your project name here ---
PROJECT_NAME = "qol-immigration"  # UPDATE per project

config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)
print(f"Project: {config.get('title', PROJECT_NAME)}")
print(f"Processed data: {processed_dir}")

Project: Quality of Life in the USA vs. Immigration Trends
Processed data: /Users/matt/data-adventures/projects/qol-immigration/data/processed


In [2]:
# Load the three analysis-ready panels from the merge step
PANELS = ["state_panel", "national_panel", "district_crosssection"]
datasets = {}
for name in PANELS:
    path = processed_dir / f"{name}.parquet"
    if path.exists():
        datasets[name] = pd.read_parquet(path)
        print(f"{name}: {datasets[name].shape}")

if not datasets:
    raise RuntimeError("No processed data found. Run 'python run.py qol-immigration --stage data' first.")

# Curate state panel to key indicators (drop redundant totals/duplicates)
STATE_INDICATORS = [
    "year", "state_fips", "state_name", "state_abbr",
    "median_household_income", "poverty_pct", "real_gdp_millions",
    "hpi_annual_avg", "life_expectancy",
    "acgr_total", "bachelors_pct",
    "foreign_born_pct", "lpr_per_million", "refugees_per_million",
    "naturalizations_total", "asylees_total",
    "voter_reg_total_pct",
]
available = [c for c in STATE_INDICATORS if c in datasets["state_panel"].columns]
datasets["state_panel"] = datasets["state_panel"][available].copy()
print(f"\nstate_panel curated to {len(available) - 4} indicators (dropped redundant columns)")

# Coverage overview
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    if "year" in df.columns:
        print(f"  Years: {int(df['year'].min())}–{int(df['year'].max())}")
    if "state_name" in df.columns:
        print(f"  States: {df['state_name'].nunique()}")
    num_cols = df.select_dtypes(include="number").columns.drop("year", errors="ignore")
    non_null = df[num_cols].notna().mean().sort_values()
    print(f"  Sparsest indicators (% non-null):")
    for col in non_null.head(5).index:
        print(f"    {col}: {non_null[col]:.1%}")

state_panel: (2652, 25)


national_panel: (79, 5)
district_crosssection: (448, 51)

state_panel curated to 13 indicators (dropped redundant columns)

--- state_panel ---
  Years: 1972–2025
  States: 51
  Sparsest indicators (% non-null):
    life_expectancy: 5.8%
    acgr_total: 17.1%
    lpr_per_million: 21.2%
    refugees_per_million: 21.2%
    naturalizations_total: 21.2%

--- national_panel ---
  Years: 1947–2025
  Sparsest indicators (% non-null):
    poverty_rate: 40.5%
    median_household_income_real: 51.9%
    median_weekly_earnings_real: 59.5%
    gini_coefficient: 98.7%

--- district_crosssection ---
  States: 50
  Sparsest indicators (% non-null):
    FiveThirtyEight Partisan Lean (2021 Version): 0.0%
    FiveThirtyEight Partisan Lean (2020 Version): 0.0%
    Cook PVI: 0.0%
    Geography ID #: 0.0%
    Party of Representative (D or R): 0.0%


## Univariate Profiles

In [3]:
from pipeline.analyze import univariate_profiles

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    profiles = univariate_profiles(df)
    display(profiles)


--- state_panel ---


,count,mean,std,min,5%,25%,50%,75%,95%,max,missing_pct,zeros_pct,outliers_low,outliers_high
year,2652.0,1999.4615,15.0789,1972.0000,1976.0000,1986.7500,1999.500,2012.2500,2023.000,2025.000,0.00,0.00,0,0
median_household_income,918.0,57256.7963,13090.1044,32938.0000,40547.5500,47188.7500,55113.000,65023.2500,82266.600,108210.000,65.38,0.00,0,16
poverty_pct,918.0,13.5554,3.1780,7.0800,9.2565,11.1125,13.110,15.8975,19.086,24.150,65.38,0.00,0,2
real_gdp_millions,1428.0,341578.8996,431002.6601,21372.2000,37896.7950,83620.4750,205952.650,420563.3500,1233769.295,3306928.600,46.15,0.00,0,92
hpi_annual_avg,2601.0,270.8150,185.7319,43.6375,76.5225,125.9425,227.290,348.9100,642.735,1287.865,1.92,0.00,0,110
life_expectancy,153.0,76.9471,2.2337,70.9000,72.6200,75.5000,77.400,78.6000,80.040,80.900,94.23,0.00,0,0
acgr_total,453.0,83.3625,5.6336,59.0000,71.7600,80.4000,84.500,87.5000,90.200,92.100,82.92,0.00,12,0
bachelors_pct,765.0,30.8547,6.7495,17.0900,21.7840,26.3500,29.920,34.6000,42.084,65.940,71.15,0.00,0,16
foreign_born_pct,918.0,9.0403,6.0882,1.0900,2.1800,4.2475,6.980,13.3600,21.487,27.420,65.38,0.00,0,10
lpr_per_million,561.0,2217.6257,1451.5512,276.6550,551.1540,1165.1250,1814.780,2981.5270,5384.854,8143.153,78.85,0.00,0,20



--- national_panel ---


,count,mean,std,min,5%,25%,50%,75%,95%,max,missing_pct,zeros_pct,outliers_low,outliers_high
year,79.0,1986.0000,22.9492,1947.000,1950.9000,1966.5000,1986.0000,2005.5000,2021.1000,2025.000,0.00,0.0,0,0
gini_coefficient,78.0,0.3990,0.0390,0.348,0.3527,0.3622,0.3905,0.4375,0.4563,0.462,1.27,0.0,0,0
median_household_income_real,41.0,70423.9024,6254.0574,60420.000,62700.0000,65440.0000,70040.0000,72790.0000,82690.0000,83730.000,48.10,0.0,0,0
poverty_rate,32.0,13.3906,1.3069,11.300,11.8100,12.5000,13.1500,14.0750,15.8450,15.900,59.49,0.0,0,0
median_weekly_earnings_real,47.0,334.1472,18.3634,311.500,312.9000,317.5000,333.5000,342.0000,369.3750,380.250,40.51,0.0,0,1



--- district_crosssection ---


,count,mean,std,min,5%,25%,50%,75%,95%,max,missing_pct,zeros_pct,outliers_low,outliers_high
FiveThirtyEight Partisan Lean (2021 Version),0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.0,0,0
FiveThirtyEight Partisan Lean (2020 Version),0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.0,0,0
Cook PVI,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.0,0,0
Biden %,435.0,51.767816,16.0817,18.0,27.7,40.0,50.0,63.0,80.0,91.0,2.90,0.0,0,0
Trump %,435.0,46.606897,16.040256,8.0,19.0,35.0,48.0,59.0,71.0,81.0,2.90,0.0,0,0
2020 Pres. Margin,435.0,5.16092,32.148874,-63.0,-44.0,-19.0,2.0,29.0,60.3,83.0,2.90,0.89,0,0
Clinton %,435.0,49.128736,17.056614,17.0,25.0,36.0,47.0,61.0,80.3,94.0,2.90,0.0,0,0
Trump %.1,435.0,45.937931,16.817489,5.0,17.0,35.0,48.0,58.0,70.3,80.0,2.90,0.0,0,0
2016 Pres. Margin,435.0,3.204598,33.800403,-63.0,-46.0,-23.0,-2.0,26.0,62.6,89.0,2.90,0.22,0,0
Obama %,435.0,51.508046,15.755821,19.0,30.0,39.0,49.0,61.5,81.0,97.0,2.90,0.0,0,1


## Time Series Trends

In [4]:
from pipeline.analyze import time_series_trends

# Specify the time column name used in your data
TIME_COL = "year"  # UPDATE if your data uses a different column

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        print(f"{name}: no '{TIME_COL}' column, skipping trends.")
        continue
    print(f"\n--- {name} ---")
    trends = time_series_trends(df, time_col=TIME_COL)
    display(trends["summary"])
    for fig in trends["figures"]:
        fig.show()


--- state_panel ---


,indicator,direction,slope_per_year,r_squared,p_value,start_value,end_value
0,median_household_income,rising,1558.038982,0.8974,0.000000,45840.4902,77588.9020
1,poverty_pct,falling,-0.066858,0.1165,0.165774,13.0731,12.1978
2,real_gdp_millions,rising,7012.639314,0.9771,0.000000,241831.9549,454543.5627
3,hpi_annual_avg,rising,10.227203,0.8882,0.000000,63.5029,696.6308
4,life_expectancy,falling,-1.139216,0.9328,0.166877,78.2627,75.9843
5,acgr_total,rising,0.619994,0.9887,0.000000,79.7500,85.7469
6,bachelors_pct,rising,0.579716,0.9826,0.000000,27.3537,35.7647
7,foreign_born_pct,rising,0.091012,0.9645,0.000000,8.1908,9.9729
8,lpr_per_million,falling,-41.756680,0.1556,0.229856,2272.7053,2437.8153
9,refugees_per_million,falling,-17.379583,0.4480,0.024290,223.0520,189.0652



--- national_panel ---


,indicator,direction,slope_per_year,r_squared,p_value,start_value,end_value
0,gini_coefficient,rising,0.001578,0.8393,0.000000,0.376,0.4560
1,median_household_income_real,rising,460.533101,0.7781,0.000000,60420.000,83730.0000
2,poverty_rate,rising,0.006145,0.0021,0.804884,12.800,12.1000
3,median_weekly_earnings_real,rising,1.158486,0.7482,0.000000,331.500,373.6667


district_crosssection: no 'year' column, skipping trends.


## Changepoint Detection

In [5]:
from pipeline.analyze import detect_changepoints

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    cp_results = detect_changepoints(df, time_col=TIME_COL)
    display(cp_results["summary"])
    for fig in cp_results["figures"]:
        fig.show()


--- state_panel ---


,indicator,changepoint_at,mean_before,mean_after,shift_magnitude
0,median_household_income,2015,50703.4745,65448.4485,14744.9740
1,real_gdp_millions,2007,282245.3502,374541.9826,92296.6324
2,real_gdp_millions,2017,312131.0667,415198.4819,103067.4152
3,hpi_annual_avg,2000,139.4007,397.1748,257.7741
4,hpi_annual_avg,2020,227.4661,595.9311,368.4650
5,bachelors_pct,2018,29.2266,34.1107,4.8840
6,foreign_born_pct,2015,8.6537,9.5236,0.8699



--- national_panel ---


,indicator,changepoint_at,mean_before,mean_after,shift_magnitude
0,gini_coefficient,1982,0.3615,0.4296,0.0681
1,gini_coefficient,1992,0.3680,0.4413,0.0733
2,gini_coefficient,2012,0.3880,0.4545,0.0665
3,median_household_income_real,1999,64636.6667,73762.6923,9126.0256
4,median_household_income_real,2019,68438.5714,82005.0000,13566.4286
5,median_weekly_earnings_real,1999,318.0000,346.1080,28.1080
6,median_weekly_earnings_real,2019,328.1500,368.4167,40.2667


## Pairwise Correlations

In [6]:
from pipeline.analyze import pairwise_correlations

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    corr_results = pairwise_correlations(df)
    display(corr_results["summary"])
    corr_results["heatmap"].show()


--- state_panel ---


,var_a,var_b,pearson_r,spearman_r,abs_max
0,real_gdp_millions,naturalizations_total,0.9420,0.9409,0.9420
1,year,hpi_annual_avg,0.8107,0.9214,0.9214
2,foreign_born_pct,lpr_per_million,0.9171,0.9190,0.9190
3,naturalizations_total,asylees_total,0.8546,0.8683,0.8683
4,median_household_income,bachelors_pct,0.8092,0.8552,0.8552
5,real_gdp_millions,asylees_total,0.8289,0.8430,0.8430
6,hpi_annual_avg,bachelors_pct,0.7761,0.8059,0.8059
7,median_household_income,hpi_annual_avg,0.7761,0.7536,0.7761
8,poverty_pct,life_expectancy,-0.7361,-0.7178,0.7361
9,foreign_born_pct,naturalizations_total,0.7267,0.7279,0.7279



--- national_panel ---


,var_a,var_b,pearson_r,spearman_r,abs_max
0,median_household_income_real,median_weekly_earnings_real,0.9176,0.8275,0.9176
1,year,gini_coefficient,0.9161,0.8749,0.9161
2,year,median_household_income_real,0.8821,0.8561,0.8821
3,year,median_weekly_earnings_real,0.8650,0.8715,0.8715
4,gini_coefficient,median_weekly_earnings_real,0.7251,0.8447,0.8447
5,gini_coefficient,median_household_income_real,0.7821,0.8101,0.8101
6,median_household_income_real,poverty_rate,-0.4862,-0.5449,0.5449



--- district_crosssection ---


,var_a,var_b,pearson_r,spearman_r,abs_max
0,district_number,district_number_cis,1.0000,1.0000,1.0000
1,Romney %,2012 Pres. Margin,-0.9996,-0.9995,0.9996
2,Trump %,2020 Pres. Margin,-0.9996,-0.9996,0.9996
3,Biden %,2020 Pres. Margin,0.9996,0.9996,0.9996
4,Obama %,2012 Pres. Margin,0.9995,0.9995,0.9995
...,...,...,...,...,...
216,2016 Pres. Margin,U.S. Citizens 18+ Population,-0.5117,-0.5022,0.5117
217,Clinton %,U.S. Citizens 18+ Population,-0.5110,-0.4934,0.5110
218,white_cvap_pct,Democratic share of two party vote,-0.5085,-0.5098,0.5098
219,Trump %.1,U.S. Citizens 18+ Population,0.5078,0.5026,0.5078


## Geographic Variation

In [7]:
from pipeline.analyze import geographic_variation

GEO_COL = "state_name"

for name, df in datasets.items():
    if GEO_COL not in df.columns:
        print(f"{name}: no '{GEO_COL}' column, skipping geo variation.")
        continue
    # Drop non-indicator numeric columns before geographic analysis
    analysis_df = df.drop(columns=["year", "state_fips"], errors="ignore")
    print(f"\n--- {name} ---")
    geo_results = geographic_variation(analysis_df, geo_col=GEO_COL)
    display(geo_results["summary"])
    for fig in geo_results["figures"]:
        fig.show()


--- state_panel ---


,indicator,region,value,z_score,direction
0,median_household_income,Maryland,7.669372e+04,2.0858,high
1,poverty_pct,Mississippi,2.106560e+01,2.5681,high
2,real_gdp_millions,California,2.334077e+06,4.7308,high
3,real_gdp_millions,New York,1.451934e+06,2.6363,high
4,real_gdp_millions,Texas,1.424940e+06,2.5722,high
5,hpi_annual_avg,District of Columbia,4.192764e+02,2.1260,high
6,hpi_annual_avg,Massachusetts,4.859186e+02,3.0804,high
7,hpi_annual_avg,New York,4.306699e+02,2.2892,high
8,life_expectancy,Mississippi,7.240000e+01,-2.2957,low
9,life_expectancy,West Virginia,7.276670e+01,-2.1106,low


national_panel: no 'state_name' column, skipping geo variation.

--- district_crosssection ---


,indicator,region,value,z_score,direction
0,Obama %,Hawaii,7.050000e+01,2.1498,high
1,Obama %,Utah,2.475000e+01,-2.4060,low
2,Romney %,Hawaii,2.800000e+01,-2.1432,low
3,Romney %,Utah,7.250000e+01,2.3070,high
4,2012 Pres. Margin,Hawaii,4.300000e+01,2.1662,high
5,2012 Pres. Margin,Utah,-4.800000e+01,-2.3596,low
6,Obama %.1,Hawaii,7.150000e+01,2.2419,high
7,McCain %,Hawaii,2.650000e+01,-2.2771,low
8,2008 Pres. Margin,Hawaii,4.550000e+01,2.2897,high
9,fte_elasticity,Alabama,8.100000e-01,-2.1093,low
